## MAGE v.1.1 - Gene expression data processing using DESeq2
---

### *Table of contents*
- [Background](#background)
- [Set-up and data preprocessing](#set-up-and-data-preprocessing)
- [Expression matrices](#expression-matrices)
- [Differential expression analysis](#differential-expression-analysis)
- [Environment](#environment)

### *Background*

This notebook details the steps taken to process gene expression data (measured using `salmon` and described [here](https://github.com/mccoy-lab/MAGE/blob/2ad80f736d6c6932e746456300cbd82c68092032/analysis_pipeline/01_data_preparation/expression_quantification/README.md)) using the `DESeq2` package in R. The products of this notebook are the following:
- Raw count, estimated mean ($\mu$), and variance-stabilizing transformed (VST) expression matrices.
- Differential expression summary statistics for each continental group label.

The pipeline detailed below follows the best-practices described in the `DESeq2` manual, which can be found [here](https://bioconductor.org/packages/devel/bioc/vignettes/DESeq2/inst/doc/DESeq2.html).

### *Set-up and data preprocessing*

The data pre-requisites for this pipeline are the following:
- A directory of `salmon` quantification files, one per sample, containing the estimated counts for each gene.
- A metadata file describing the biological (e.g., *continental group* and *sex* labels) and technical (e.g., *batch*) covariates for each sample.
- A `.bed` file containing a list of genes to be included in the analysis. Here, we use a list of genes which meet experiment-wide expression thresholds (described [here](https://github.com/mccoy-lab/MAGE/blob/2ad80f736d6c6932e746456300cbd82c68092032/analysis_pipeline/01_data_preparation/expression_quantification/README.md)).
- A reference genome annotation file (e.g., `.gtf`) for creating a `tximport` transcript-to-gene mapping object (for gene-level analysis).

First, we will load the required libraries:

In [4]:
# Load required libraries
invisible(
  c(library(DESeq2),
    library(tidyverse),
    library(yaml))
)

Next, we will load the data prerequisites outlined above. Paths to these files are stored in `03_deseq_config.yaml`.

In [ ]:
# Load data prerequisities
## Load config yaml
deseq_config <- read_yaml("03_deseq_config.yaml")

## Read in metadata
metadata <- suppressMessages(read_tsv(deseq_config$metadata))
head(metadata[1:5, 4:9])

sample_kgpID,batch,continentalGroup,population,sex,numReads
<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>
NA12843,1,EUR,CEU,XX,24858299
NA12878,1,EUR,CEU,XX,32005058
NA18536,1,EAS,CHB,XY,25187628
NA18555,1,EAS,CHB,XX,29924207
NA18877,1,AFR,YRI,XY,27877330


In [ ]:
## Read in filtered expression bed
filtered_expression_bed <- suppressMessages(
  read_tsv(deseq_config$filtered_expression_bed,
           col_select = c(1, 4))
)
head(filtered_expression_bed)

#Chr,ID
<chr>,<chr>
chr1,ENSG00000227232.5
chr1,ENSG00000233750.3
chr1,ENSG00000238009.6
chr1,ENSG00000268903.1
chr1,ENSG00000269981.1
chr1,ENSG00000241860.7
